<a href="https://colab.research.google.com/github/2403a52269-code/NLP/blob/main/NLP_14_4_2403A52269(T_Pranith_reddy).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
# Step 1: Import libraries
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

In [41]:
df = pd.read_csv("/content/sms_spam_dataset.csv", encoding='latin-1', header=None)
df = df[[0,1]]
df.columns = ['label','text']

df['label'] = df['label'].map({'ham':0, 'spam':1})
df.dropna(subset=['label'], inplace=True) # Drop rows where 'label' is NaN

In [42]:
texts = df['text']
labels = df['label']

# Step 3: Convert text to BoW
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts).toarray()
y = labels

In [43]:
# Step 4: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [44]:
# Step 5: Create Dataset class
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [45]:
train_loader = DataLoader(TextDataset(X_train, y_train.values), batch_size=2, shuffle=True)
test_loader = DataLoader(TextDataset(X_test, y_test.values), batch_size=2)

In [46]:
# Step 6: Define CNN Model (with hidden layer)
class CNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        kernels=4
        kernel_size=3
        self.conv = nn.Conv1d(1, kernels, kernel_size)

        pool_size=2
        self.pool = nn.MaxPool1d(pool_size)

        # Correctly calculate flattened size
        # Output length after convolution
        conv_output_length = input_size - kernel_size + 1
        # Output length after max pooling (default stride=pool_size)
        pool_output_length = (conv_output_length - pool_size) // pool_size + 1
        flattened_size = kernels * pool_output_length

        # Hidden layer
        hidden_neurons=8
        self.fc1 = nn.Linear(flattened_size, hidden_neurons)   # 8 neurons (you can change)

        # Output layer
        output_neurons=1
        self.fc2 = nn.Linear(hidden_neurons, output_neurons)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)              # (batch, 1, features)
        x = torch.relu(self.conv(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)       # Flatten

        x = torch.relu(self.fc1(x))     # Hidden layer + activation
        x = self.fc2(x)                 # Output layer

        return self.sigmoid(x)

In [47]:
model = CNNModel(input_size=X.shape[1])

In [48]:
# Step 7: Loss and Optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [53]:
# Step 8: Training
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch).squeeze(-1)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.0110
Epoch 2, Loss: 0.0030
Epoch 3, Loss: 0.0044
Epoch 4, Loss: 0.0056
Epoch 5, Loss: 0.0013
Epoch 6, Loss: 0.0012
Epoch 7, Loss: 0.0005
Epoch 8, Loss: 0.0006
Epoch 9, Loss: 0.0009
Epoch 10, Loss: 0.0007


In [52]:
y_pred = []
y_actual = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch).squeeze(-1) # Changed .squeeze() to .squeeze(-1)
        preds = (output > 0.5).int()
        y_pred.extend(preds.tolist())
        y_actual.extend(y_batch.tolist())

y_actual = [int(y) for y in y_actual]
print(y_pred)

[0, 0, 1, 0]


In [51]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_pred))

print("\nClassification Report:")
print(classification_report(y_actual, y_pred))

print("\nAccuracy Score:", accuracy_score(y_actual, y_pred))


Confusion Matrix:
[[2 1]
 [1 0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.67      0.67         3
           1       0.00      0.00      0.00         1

    accuracy                           0.50         4
   macro avg       0.33      0.33      0.33         4
weighted avg       0.50      0.50      0.50         4


Accuracy Score: 0.5
